In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
from tqdm import tqdm
import pandas as pd
tqdm.pandas()

class CardioNER:
    
    def __init__(self, model_name: str, iob: bool = False, device: str = "cuda"):
        
        self.model_name = model_name
        self.iob = iob
        self.device = device
        self.entities = []
        
        self.tokenizer = self.load_tokenizer(model_name)
        self.model = self.load_model(model_name)
        self.pipe = pipeline(
                        "ner",
                        model=self.model,
                        tokenizer=self.tokenizer,
                        # device=1,  # Use GPU if available
                    )
        
        if self.iob:
            self.tag2label = {
                "O": 0,
                "B-DISEASE": 1,
                "I-DISEASE": 2,
                "B-MEDICATION": 3,
                "I-MEDICATION": 4,
                "B-PROCEDURE": 5,
                "I-PROCEDURE": 6,
                "B-SYMPTOM": 7,
                "I-SYMPTOM": 8,
            }
        else:
            self.tag2label = {
                    "O": 0,
                    "DISEASE": 1,
                    "MEDICATION": 2,
                    "PROCEDURE": 3,
                    "SYMPTOM": 4,
                }
    
    def load_tokenizer(self, model_name: str):
        # Load the tokenizer from the specified model name
        try:
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            return tokenizer
        except Exception as e:
            raise ValueError(f"Error loading tokenizer: {e}")
    
    def load_model(self, model_name: str):
        # Load the model from the specified model name
        try:
            model = AutoModelForTokenClassification.from_pretrained(model_name)
            return model
        except Exception as e:
            raise ValueError(f"Error loading model: {e}")
        
    
    def extract_entities(self, text: str):
        ner_outputs = self.pipe(text).copy()
        ner_outputs = self.process_ner_results(ner_outputs, self.tag2label, iob=self.iob)
        
        entities = []
        current_entity = None
        # Create a copy of the ner_outputs to avoid modifying the original
        ner_outputs = ner_outputs.copy()
    
        for token in ner_outputs:
            if token["label"] == "O":
                continue
            label = token["label"].replace("B-", "").replace("I-", "")
            word = token["word"].replace("Ġ", "")
            score = token["score"]

            if current_entity is None:
                # Start a new entity
                current_entity = {
                    "label": label,
                    "text": word.strip(),
                    "start": token["start"],
                    "end": token["end"],
                    "score": [score],
                }
            elif current_entity["label"] == label and token["start"] == current_entity["end"]:
                # Continue current entity (must be same label and consecutive)
                current_entity["text"] += word
                current_entity["end"] = token["end"]
                current_entity["score"].append(score)
            elif current_entity["label"] == label and token["start"] == (current_entity["end"] + 1):
                # Continue current entity (must be same label and consecutive)
                current_entity["text"] += " " + word
                current_entity["end"] = token["end"]
                current_entity["score"].append(score)
            else:
                # Save current and start new
                entities.append(current_entity)
                current_entity = {
                    "label": label,
                    "text": word.strip(),
                    "start": token["start"],
                    "end": token["end"],
                    "score": [score],
                }

        if current_entity:
            entities.append(current_entity)

        # Average the scores
        for ent in entities:
            ent["score"] = sum(ent["score"]) / len(ent["score"])

        return entities
    
    def extract_entities_from_df(self, df: pd.DataFrame, text_column: str="text", filename_column: str="filenameid"):
        # Apply the entity extraction to each row in the DataFrame
        
        df_txt = df.copy()
        df_ner = df_txt.set_index(filename_column).progress_apply(lambda x: self.extract_entities(x[text_column]), axis=1)
        df_ner_exp = df_ner.explode().reset_index()
        df_ner_exp = df_ner_exp.rename(columns={0: "entities"})
        df_ner_exp["mention_class"] = df_ner_exp["entities"].apply(lambda x: x["label"])
        df_ner_exp["span"] = df_ner_exp["entities"].apply(lambda x: x["text"])
        df_ner_exp["start"] = df_ner_exp["entities"].apply(lambda x: x["start"])
        df_ner_exp["end"] = df_ner_exp["entities"].apply(lambda x: x["end"])
        df_ner_exp["score"] = df_ner_exp["entities"].apply(lambda x: x["score"])
        df_ner_exp = df_ner_exp.drop(columns=["entities"])

        return df_ner_exp
            
    @staticmethod
    def process_ner_results(ner_results, tag2label, iob=False):

        label2tag = {v: k for k, v in tag2label.items()}

        proc_results = []
        for result in ner_results:
            if result['entity'] != 'O':
                proc_results.append({
                    'start': result['start'],
                    'end': result['end'],
                    'label': label2tag[int(result['entity'].split('_')[-1])],
                    'score': result['score'],
                    'word': result['word'],
                })
                
        return proc_results
    


/gpfs/projects/bsc14/code/nlp4bia/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
model_name = "/gpfs/projects/bsc14/code/CardioNER/output/es/NOIOB_cpt_biomed_roberta_token/E120BS32LR2e5/hf"

ner_system = CardioNER(model_name=model_name, iob=False, device="cuda")

Device set to use cuda:0


In [34]:
tok = ner_system.tokenizer

example1 = """El varón tiene fiebre, tos y dificultad respiratoria."""

for token in tok.encode(example1):
    print(token, "\t", tok.decode(token))

0 	 <s>
470 	  El
8276 	  varón
861 	  tiene
2389 	  fiebre
15 	 ,
3084 	  tos
290 	  y
4133 	  dificultad
3516 	  respiratoria
17 	 .
2 	 </s>


In [33]:
tok2 = AutoTokenizer.from_pretrained("PlanTL-GOB-ES/bsc-bio-ehr-es")
for token in tok2.encode(example1):
    print(token, "\t", tok2.decode(token))

0 	 <s>
458 	  El
8452 	  varón
862 	  tiene
2547 	  fiebre
15 	 ,
3272 	  tos
290 	  y
4227 	  dificultad
3694 	  respiratoria
17 	 .
2 	 </s>


In [21]:
example1 = """
El varón tiene fiebre,tos y dificultad respiratoria.
"""

ner_system.extract_entities(example1)[:10]

[{'label': 'SYMPTOM',
  'text': 'fiebre tos',
  'start': 16,
  'end': 26,
  'score': np.float32(0.3731606)},
 {'label': 'SYMPTOM',
  'text': 'dificultad respiratoria',
  'start': 29,
  'end': 52,
  'score': np.float32(0.46415663)}]

In [10]:
from nlp4bia.datasets.benchmark.symptemist import SymptemistLoader, SymptemistGazetteer

symptemist_loader = SymptemistLoader()
df_symp = symptemist_loader.df

df_symp

preprocessing data...


,filenameid,mention_class,span,code,sem_rel,is_abbreviation,is_composite,needs_context,extension_esp,text,split
0,es-S0365-66912011000600005-2#333#361,SINTOMA,«manchas» en el campo visual,246658005,EXACT,None,False,False,None,Varón de 37 años ex-adicto a drogas por vía pa...,train
1,es-S0004-06142010000300011-1#649#716,SINTOMA,5HIAA en orina de 24 horas estaba dentro de lo...,171250001,NARROW,None,False,False,None,Paciente de 24 años con un hermano gemelo que ...,train
2,es-S1130-01082007000700011-2#1463#1505,SINTOMA,A nivel analítico no presentaba alteración,166315009,NARROW,None,False,False,None,"Mujer de 68 años, como antecedentes personales...",train
3,es-S0210-48062009000300017-1#2713#2759,SINTOMA,a nivel del cardias masa mamelonada y ulcerada,126825008,NARROW,None,False,False,None,Paciente de 57 años de edad remitido a urgenci...,train
4,es-S1130-01082006000100014-1#2282#2295,SINTOMA,abdomen agudo,9209005,EXACT,None,False,False,None,"Se trata de una mujer de 35 años, con antecede...",train
...,...,...,...,...,...,...,...,...,...,...,...
12009,es-S1130-01082007001100009-1#393#446,SINTOMA,zona indurada en la pared lateral izquierda de...,5964004,NARROW,None,False,False,None,Mujer de 42 años estudiada en Consultas de Gas...,test
12010,es-S0212-71992005000600008-1#903#997,SINTOMA,"zona periumbilical, donde se aprecia a la insp...",448569009,NARROW,None,False,False,None,"Varón de 71 años, que ingresó en el servicio d...",test
12011,es-S1139-76322016000300008-1#1642#1699,SINTOMA,zona superior de la vejiga hacia el ombligo un...,235993005,NARROW,None,False,False,None,Presentamos el caso de una lactante de cinco m...,test
12012,es-S1134-80462005000300004-1#2808#2869,SINTOMA,zonas hipoestésicas y espásticas en ambos miem...,NO_CODE,NO_CODE,None,False,False,None,Presentamos el caso de un varón de 53 años de ...,test


In [13]:
df_txt = df_symp[["filenameid", "text", "split"]].copy()
df_txt["filenameid"] = df_txt["filenameid"].str.split("#").str[0]
df_txt = df_txt.drop_duplicates().reset_index(drop=True)
df_txt

,filenameid,text,split
0,es-S0365-66912011000600005-2,Varón de 37 años ex-adicto a drogas por vía pa...,train
1,es-S0004-06142010000300011-1,Paciente de 24 años con un hermano gemelo que ...,train
2,es-S1130-01082007000700011-2,"Mujer de 68 años, como antecedentes personales...",train
3,es-S0210-48062009000300017-1,Paciente de 57 años de edad remitido a urgenci...,train
4,es-S1130-01082006000100014-1,"Se trata de una mujer de 35 años, con antecede...",train
...,...,...,...
985,es-S1698-69462006000200016-1,Presentamos el caso de una paciente de 20 años...,test
986,es-S1138-123X2004000100006-2,"Paciente de siete años de edad, en etapa de de...",test
987,es-S1130-01082008000900011-1,Se trata de un varón de 39 años de edad remiti...,test
988,es-S1130-05582008000600006-1,Enfermo de 19 años que ingresa procedente de u...,test


In [14]:
df_ner = ner_system.extract_entities_from_df(df_txt, text_column="text", filename_column="filenameid")
df_ner

100%|██████████| 990/990 [00:13<00:00, 73.33it/s]


,filenameid,mention_class,span,start,end,score
0,es-S0365-66912011000600005-2,DISEASE,adicto a drogas por vÃŃa parenteral,20,54,0.776217
1,es-S0365-66912011000600005-2,DISEASE,hepatitis crÃ³nica por virus C genotipo 4,74,114,0.952462
2,es-S0365-66912011000600005-2,MEDICATION,IFN pegilado alpha 2,184,204,0.722213
3,es-S0365-66912011000600005-2,MEDICATION,ribavirina,223,233,0.891714
4,es-S0365-66912011000600005-2,SYMPTOM,visiÃ³n borrosa,316,330,0.944249
...,...,...,...,...,...,...
36181,es-S1130-05582017000300150-3,PROCEDURE,hemimandibulectomÃŃa,249,268,0.967036
36182,es-S1130-05582017000300150-3,PROCEDURE,colocaciÃ³n de prÃ³tesis de reconstrucciÃ³n,282,322,0.842621
36183,es-S1130-05582017000300150-3,PROCEDURE,cÃ³ndilo,335,342,0.819148
36184,es-S1130-05582017000300150-3,PROCEDURE,RadiografÃŃa,344,355,0.936702


In [15]:
df_symp_pred = df_ner[df_ner["mention_class"].isin(["SYMPTOM"])].reset_index(drop=True)
df_symp_pred

,filenameid,mention_class,span,start,end,score
0,es-S0365-66912011000600005-2,SYMPTOM,visiÃ³n borrosa,316,330,0.944249
1,es-S0365-66912011000600005-2,SYMPTOM,manchasÂ» en el campo visual,334,361,0.900879
2,es-S0365-66912011000600005-2,SYMPTOM,exudados algodonosos en polo posterior,495,533,0.806906
3,es-S0365-66912011000600005-2,SYMPTOM,exudado algodonoso en reabsorciÃ³n inferior a ...,742,792,0.734187
4,es-S0004-06142010000300011-1,SYMPTOM,diarrea,542,549,0.479284
...,...,...,...,...,...,...
10953,es-S1130-05582008000600006-1,SYMPTOM,PÃ©rdida cutÃ¡neo-mucosa,335,357,0.540223
10954,es-S1130-05582008000600006-1,SYMPTOM,de regiÃ³n yugal,368,383,0.541721
10955,es-S1130-05582008000600006-1,SYMPTOM,mejilla,396,403,0.624659
10956,es-S1130-05582008000600006-1,SYMPTOM,retracciÃ³n,920,930,0.459690


In [ ]:
df_symp_comp = df_symp[["filenameid", "span"]].copy()
df_symp_comp["start"] = df_symp_comp["filenameid"].str.split("#").str[1].astype(int)
df_symp_comp["end"] = df_symp_comp["filenameid"].str.split("#").str[2].astype(int)
df_symp_comp["filenameid"] = df_symp_comp["filenameid"].str.split("#").str[0]

df_symp_comp = df_symp_comp.merge(df_symp_pred[["filenameid", "span", "start", "end"]], on=["filenameid", "start", "end"], 
                                  how="outer", indicator=True, suffixes=("", "_pred"))
df_symp_comp

,filenameid,span,start,end,span_pred,_merge
0,S0004-06142005000500011-1,hematuria macroscópica postmiccional,444,480,NaN,left_only
1,S0004-06142005000500011-1,microhematuria,498,512,NaN,left_only
2,S0004-06142005000500011-1,micciones normales,545,563,micciones normales,both
3,S0004-06142005000500011-1,buen estado general,602,621,buen estado general,both
4,S0004-06142005000500011-1,abdomen y genitales normales,627,655,abdomen y genitales normales,both
...,...,...,...,...,...,...
17905,es-S2340-98942015000100005-1,vómitos,950,957,vÃ³mitos,both
17906,es-S2340-98942015000100005-1,pérdida de conocimiento,959,982,pÃ©rdida de conocimiento,both
17907,es-S2340-98942015000100005-1,disnea,1542,1548,disnea,both
17908,es-S2340-98942015000100005-1,sudoración,1550,1560,sudoraciÃ³n,both


In [18]:
df_symp_comp[df_symp_comp["filenameid"] == "es-S2340-98942015000100005-1"]

,filenameid,span,start,end,span_pred,_merge
17901,es-S2340-98942015000100005-1,asintomática,360,372,NaN,left_only
17902,es-S2340-98942015000100005-1,elevación de CA 125,413,432,elevaciÃ³n de CA 125,both
17903,es-S2340-98942015000100005-1,imagen adyacente al colon,490,515,imagen adyacente al colon,both
17904,es-S2340-98942015000100005-1,disnea,942,948,disnea,both
17905,es-S2340-98942015000100005-1,vómitos,950,957,vÃ³mitos,both
17906,es-S2340-98942015000100005-1,pérdida de conocimiento,959,982,pÃ©rdida de conocimiento,both
17907,es-S2340-98942015000100005-1,disnea,1542,1548,disnea,both
17908,es-S2340-98942015000100005-1,sudoración,1550,1560,sudoraciÃ³n,both
17909,es-S2340-98942015000100005-1,libre de enfermedad,1912,1931,NaN,left_only


In [19]:
# COmpute precision, recall and f1

tp = len(df_symp_comp[(df_symp_comp["_merge"] == "both")])
fp = len(df_symp_comp[(df_symp_comp["_merge"] == "right_only")])
fn = len(df_symp_comp[(df_symp_comp["_merge"] == "left_only")])

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"TP: {tp}, FP: {fp}, FN: {fn}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1: {f1:.4f}")

TP: 5062, FP: 5896, FN: 6952
Precision: 0.4619
Recall: 0.4213
F1: 0.4407


In [37]:
df_txt.explode("ner", ignore_index=True)

,filenameid,text,split,ner
0,es-S0365-66912011000600005-2,Varón de 37 años ex-adicto a drogas por vía pa...,train,"{'label': 'DISEASE', 'text': 'ex adicto a drog..."
1,es-S0365-66912011000600005-2,Varón de 37 años ex-adicto a drogas por vía pa...,train,"{'label': 'DISEASE', 'text': 'hepatitis crÃ³ni..."
2,es-S0365-66912011000600005-2,Varón de 37 años ex-adicto a drogas por vía pa...,train,"{'label': 'DISEASE', 'text': 'carga viral', 's..."
3,es-S0365-66912011000600005-2,Varón de 37 años ex-adicto a drogas por vía pa...,train,"{'label': 'MEDICATION', 'text': 'IFN pegilado ..."
4,es-S0365-66912011000600005-2,Varón de 37 años ex-adicto a drogas por vía pa...,train,"{'label': 'MEDICATION', 'text': 'ribavirina', ..."
...,...,...,...,...
35289,es-S1130-05582017000300150-3,Lesión radiolúcida en rama y parte de cuerpo m...,test,"{'label': 'PROCEDURE', 'text': 'colocaciÃ³n', ..."
35290,es-S1130-05582017000300150-3,Lesión radiolúcida en rama y parte de cuerpo m...,test,"{'label': 'PROCEDURE', 'text': 'prÃ³tesis de r..."
35291,es-S1130-05582017000300150-3,Lesión radiolúcida en rama y parte de cuerpo m...,test,"{'label': 'PROCEDURE', 'text': 'cÃ³ndilo', 'st..."
35292,es-S1130-05582017000300150-3,Lesión radiolúcida en rama y parte de cuerpo m...,test,"{'label': 'PROCEDURE', 'text': 'RadiografÃŃa',..."


In [ ]:
df_txt["ner_results"].

0                 label  ...     score
0      DISEASE  ...
1                 label  ...     score
0      DISEASE  ...
2                label  ...     score
0     DISEASE  .....
3                label  ...     score
0     SYMPTOM  .....
4                 label  ...     score
0      DISEASE  ...
                               ...                        
11043             label  ...     score
0      SYMPTOM  ...
11048            label                                 ...
11084             label  ...     score
0    PROCEDURE  ...
11148            label  ...     score
0     DISEASE  .....
11634            label                          text  s...
Name: ner_results, Length: 990, dtype: object

In [8]:
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

def ner_outputs(text):
    ner_results_1 = pipe(text)

    proc_results_1 = process_ner_results(ner_results_1)
    results_1 = assemble_ner_output_flat(proc_results_1)

    df_res_1 = pd.DataFrame(results_1)[["label", "text"]].drop_duplicates()
    return df_res_1

def ner_output_dataset(df, col_text_name, fileid_col="filenameid"):
    ls_cc_ents = []
    for _, row in tqdm(df.iterrows(), total=len(df_pairs)):
        text = row[col_text_name]
        filenameid = row[fileid_col]
        df_res_1 = ner_outputs(text)
        df_res_1["filenameid"] = filenameid
        ls_cc_ents.append(df_res_1)
    
    return pd.concat(ls_cc_ents, ignore_index=True)
        

df_cc_ents = ner_output_dataset(df_pairs, "clinical_case")
df_ds_ents = ner_output_dataset(df_pairs, "discharge_summary")

  1%|█                                                                                                                                                      | 7/1000 [00:00<00:15, 63.69it/s]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:15<00:00, 64.47it/s]


In [9]:
ents_dist_cc = df_cc_ents.label.value_counts()
print(ents_dist_cc)
print("TOTAL:", ents_dist_cc.sum())

label
PROCEDURE     11338
DISEASE       10090
SYMPTOM        9889
MEDICATION     2135
Name: count, dtype: int64
TOTAL: 33452


In [10]:
ents_dist_ds = df_ds_ents.label.value_counts()
print(ents_dist_ds)
print("TOTAL:", ents_dist_ds.sum())

label
PROCEDURE     10790
DISEASE       10253
SYMPTOM        9255
MEDICATION     1982
Name: count, dtype: int64
TOTAL: 32280


In [11]:
df_cc_ents[df_cc_ents["label"] == "DISEASE"].text.value_counts().sort_values(ascending=False)

text
heart failure                        227
hypertension                         169
cardiomegaly                         110
atrial fibrillation                   85
cardiogenic shock                     65
                                    ... 
-se                                    1
coronary steal syndrome from           1
nephritic syndrome                     1
generalised tonic-clonic seizures      1
Severe lice infestation                1
Name: count, Length: 5123, dtype: int64

In [12]:
from sentence_transformers import SentenceTransformer
from src.search_index import FaissIndex

model_st = SentenceTransformer("/gpfs/projects/bsc14/abecerr1/hub/models--cambridgeltl--SapBERT-from-PubMedBERT-fulltext-mean-token/snapshots/9f95c2e962719c70f25bf7a1f33bd8d9e9448750", device="cuda:1")

No sentence-transformers model found with name /gpfs/projects/bsc14/abecerr1/hub/models--cambridgeltl--SapBERT-from-PubMedBERT-fulltext-mean-token/snapshots/9f95c2e962719c70f25bf7a1f33bd8d9e9448750. Creating a new one with mean pooling.


In [13]:
d_gazetteers = {
    "PROCEDURE": "data/4_gazetteers/en/english_procedures_gazetteer.tsv",
    "DISEASE": "data/4_gazetteers/en/english_diseases_gazetteer.tsv",
    "MEDICATION": "data/4_gazetteers/en/english_medications_gazetteer.tsv",
    "SYMPTOM": "data/4_gazetteers/en/english_symptoms_gazetteer.tsv",
}

def link_ents_to_gazetteers(df, ls_label, model):

    ls_out = []
    for label in ls_label:
        print("LABEL:", label)
        df_label = df[df.label == label].copy()
        faiss_index = FaissIndex(model=model_st, gazetteer_path=d_gazetteers[label], random_seed=0)
        faiss_index.generate_search_index()
        query_embs_1 = model_st.encode(df_label["text"].tolist())
        _, I1 = faiss_index.search(query_embs_1, k=10)
        
        ls_codes = [[faiss_index.get_code_by_index(i) for i in row] for row in I1]
        ls_terms = [[faiss_index.get_term_by_index(i) for i in row] for row in I1]
        
        df_label["codes"] = ls_codes
        df_label["terms"] = ls_terms
        
        ls_out.append(df_label)
    
    return pd.concat(ls_out, ignore_index=True)

df_cc_ents_gazetteers = link_ents_to_gazetteers(df_cc_ents, ["DISEASE", "MEDICATION", "PROCEDURE", "SYMPTOM"], model_st)

        

LABEL: DISEASE


Batches:   0%|          | 0/5892 [00:00<?, ?it/s]

LABEL: MEDICATION


Batches:   0%|          | 0/3232 [00:00<?, ?it/s]

LABEL: PROCEDURE


Batches:   0%|          | 0/7141 [00:00<?, ?it/s]

LABEL: SYMPTOM


Batches:   0%|          | 0/7099 [00:00<?, ?it/s]

In [14]:
df_cc_ents_gazetteers

,label,text,filenameid,codes,terms
0,DISEASE,Amyloid light-chain amyloidosis,33175723_1,"[23132008, 23132008, 23132008, 426598005, 2749...","[Amyloid light-chain amyloidosis, Primary amyl..."
1,DISEASE,cardiac amyloidosis,33175723_1,"[16573007, 16573007, 17602002, 1187540008, 118...","[Cardiac amyloidosis, Senile cardiac amyloidos..."
2,DISEASE,cardiomyopathy,33175723_1,"[85898001, 35728003, 111285003, 89461002, 3990...","[Cardiomyopathy, Familial cardiomyopathy, Meta..."
3,DISEASE,heart failure,33175723_1,"[84114007, 84114007, 84114007, 42343007, 42343...","[Heart failure, Cardiac failure, HF - Heart fa..."
4,DISEASE,primary AL amyloidosis,33175723_1,"[23132008, 128817004, 190923000, 274945004, 56...","[AL amyloidosis, Primary amyloidosis, Sporadic..."
...,...,...,...,...,...
33447,SYMPTOM,Pul,37861254,"[233604007, 19829001, 205237003, 91434003, 846...","[Pneumonia, Pulmonary disease, Pneumonitis, Pu..."
33448,SYMPTOM,test and,37861254,"[252014003, 226219004, 68193004, 251644002, 25...","[Urethral test observation, Test diet, Thomas ..."
33449,SYMPTOM,results were normal,37861254,"[168500000, 408573005, 165324008, 312969002, 1...","[Radiology result normal, Imaging result norma..."
33450,SYMPTOM,enlarged main pulmonary arteries,37861254,"[93059006, 251047005, 93059006, 194892009, 194...","[Pulmonary artery dilatation, Dilatation of pu..."


In [15]:
df_ds_ents_gazetteers = link_ents_to_gazetteers(df_ds_ents, ["DISEASE", "MEDICATION", "PROCEDURE", "SYMPTOM"], model_st)


LABEL: DISEASE


Batches:   0%|          | 0/5892 [00:00<?, ?it/s]

LABEL: MEDICATION


Batches:   0%|          | 0/3232 [00:00<?, ?it/s]

LABEL: PROCEDURE


Batches:   0%|          | 0/7141 [00:00<?, ?it/s]

LABEL: SYMPTOM


Batches:   0%|          | 0/7099 [00:00<?, ?it/s]

In [16]:
df_ds_ents_gazetteers

,label,text,filenameid,codes,terms
0,DISEASE,Amyloid light-chain amyloidosis,33175723_1,"[23132008, 23132008, 23132008, 426598005, 2749...","[Amyloid light-chain amyloidosis, Primary amyl..."
1,DISEASE,Cardiac amyloidosis,33175723_1,"[16573007, 16573007, 17602002, 1187540008, 118...","[Cardiac amyloidosis, Senile cardiac amyloidos..."
2,DISEASE,Cardiomyopathy,33175723_1,"[85898001, 35728003, 111285003, 89461002, 3990...","[Cardiomyopathy, Familial cardiomyopathy, Meta..."
3,DISEASE,Heart failure,33175723_1,"[84114007, 84114007, 84114007, 42343007, 42343...","[Heart failure, Cardiac failure, HF - Heart fa..."
4,DISEASE,Primary AL amyloidosis,33175723_1,"[23132008, 128817004, 190923000, 274945004, 56...","[AL amyloidosis, Primary amyloidosis, Sporadic..."
...,...,...,...,...,...
32275,SYMPTOM,and right ventricle,37861254,"[301097002, 109425008, 448600002, 448103004, 2...","[Finding of right ventricle, Single right vent..."
32276,SYMPTOM,increased right ventricular systolic pressure,37861254,"[416158002, 461321006, 18050000, 56218007, 180...","[Right ventricular systolic dysfunction, Right..."
32277,SYMPTOM,: Normal,37861254,"[386549008, 225544001, 81323004, 125112009, 53...","[Normal appearance, Skin normal, Normal patien..."
32278,SYMPTOM,Enlarged main pulmonary arteries,37861254,"[93059006, 251047005, 93059006, 194892009, 194...","[Pulmonary artery dilatation, Dilatation of pu..."


In [17]:
df_cc_ents_gazetteers.to_csv("nbs/evaluation/automatic/cardioner_entities/NOIOB_EN_clinical_case_ents.tsv", sep="\t", index=False)
df_ds_ents_gazetteers.to_csv("nbs/evaluation/automatic/cardioner_entities/NOIOB_EN_discharge_summary_ents.tsv", sep="\t", index=False)